.. note::
   To download the tutorial data, use the following commands:

   **All tutorial data:**
   ```python
   from recon.data import fetch_all_tutorial_data
   fetch_all_tutorial_data(data_dir='./data')
   ```

   **Specific file (RNA data used in this tutorial):**
   ```python
   from recon.data import fetch_tutorial_data
   fetch_tutorial_data('perturbation_tuto/rna.h5ad', data_dir='./data')
   ```

# Inferring Gene Regulatory Networks with GRNBoost2 (scRNA-seq only)

This tutorial demonstrates how to use ReCoN to infer a **Gene Regulatory Network (GRN)** from **scRNA-seq data alone**, using the GRNBoost2-style regression method (via [Arboreto](https://arboreto.readthedocs.io/)) wrapped by `recon.infer_grn`.

```{tip}
Have both scRNA-seq **and** scATAC-seq data? See [Tutorial 4: Building GRNs with HuMMuS](4.recon_hummus.ipynb) instead — adding chromatin accessibility evidence generally improves GRN quality.
```

## What you will learn

1. How to compute a TF → gene network from scRNA-seq using GRNBoost2 (`compute_rna_network`)
2. Why and how to filter a GRNBoost2 network before downstream use
3. How this RNA-only GRN compares to the multilayer HuMMuS approach

## When should you use this instead of HuMMuS?

| | RNA-only (this tutorial) | HuMMuS (Tutorial 4) |
|---|---|---|
| **Input data** | scRNA-seq only | paired scRNA-seq + scATAC-seq |
| **Extra dependencies** | none (`pip install recon`) | CellOracle, bedtools, reference genome |
| **Evidence used** | TF-gene co-expression | co-expression **+** chromatin accessibility **+** motifs |
| **Speed** | fast | slower (motif scanning) |
| **Precision** | lower — co-expression only | higher — cross-validated by chromatin evidence |

```{note}
No Python 3.10 pin or CellOracle install is required here: `compute_rna_network` only depends on `arboreto` and `hummuspy`, which are installed with the base `pip install recon`.
```

In [1]:
cd ../../../

/Users/remitrimbour/ReCoN_project/ReCoN


/Users/remitrimbour/miniconda3/envs/recon-grn/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


---

## Load libraries

In [2]:
import scanpy as sc

import recon
import recon.infer_grn
import hummuspy.loader  # curated transcription factor lists

---

## Load single-cell RNA-seq data

We reuse the same scRNA-seq dataset as the [molecular treatment tutorial](1.recon_molecular_treatment.ipynb).

In [3]:
# Load scRNA-seq data
rna = sc.read_h5ad("./data/perturbation_tuto/rna.h5ad")
print(f"RNA: {rna.shape}")

RNA: (1296, 31053)


In [4]:
# Normalize, log-transform, then keep the most variable genes
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=2000)
rna = rna[:, rna.var['highly_variable']].copy()
print(f"Subset - RNA: {rna.shape}")

Subset - RNA: (1296, 2000)


/Users/remitrimbour/miniconda3/envs/recon-grn/lib/python3.10/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


---

## Compute the TF → gene network with GRNBoost2

`compute_rna_network` wraps the same gradient boosting regression (GBM) used by GRNBoost2: for each target gene, it fits a regressor against all candidate TFs and keeps the resulting feature importances as edge weights.

```{warning}
Just like GRNBoost2 edges, these TF → gene links reflect **co-expression**, not necessarily direct regulation. Without chromatin accessibility evidence to cross-validate them (as in the HuMMuS tutorial), it is especially important to filter this network before using it downstream.
```

In [5]:
# Load a curated list of mouse transcription factors
# (use "human_tfs_r_hummus" instead if your data is human)
tfs_list = hummuspy.loader.load_tfs("mouse_tfs_r_hummus")
print(f"Loaded {len(tfs_list)} transcription factors")

Loaded 901 transcription factors


In [6]:
# Compute TF -> gene network using GRNBoost2 (GBM regression)
grn = recon.infer_grn.compute_rna_network(
    rna,
    tf_names=tfs_list,
    method='GBM',
    n_cpu=4,
)
print(f"Raw GRN: {len(grn)} edges")
grn.head(3)

Calculating TF-to-gene importance


Running using 4 cores: 100%|██████████| 2000/2000 [00:26<00:00, 76.38it/s]


Raw GRN: 91331 edges


,source,target,weight
8,Nr1h3,4930578g10rik,25.948450
19,Pax5,Ighv1-36,25.637556
19,Pax5,Gm28694,25.637556


---

## Filter the network

Raw GRNBoost2 output keeps **every** TF-gene pair, most of which are noise. A common practice (also used in SCENIC/SCENIC+) is to keep, for each target gene, only its top regulators by importance, and to drop very low-weight edges genome-wide.

```{tip}
Tune `top_n_per_target` and `min_weight` to the size of your GRN — stricter filtering trades recall for precision.
```

In [7]:
top_n_per_target = 10   # keep only the strongest regulators per gene
min_weight = grn['weight'].quantile(0.5)  # drop the weaker half of edges genome-wide

filtered_grn = (
    grn[grn['weight'] >= min_weight]
    .sort_values('weight', ascending=False)
    .groupby('target', group_keys=False)
    .head(top_n_per_target)
    .reset_index(drop=True)
)

print(f"Filtered GRN: {len(filtered_grn)} edges "
      f"(from {len(grn)}), {filtered_grn['source'].nunique()} TFs, "
      f"{filtered_grn['target'].nunique()} target genes")
filtered_grn.head(10)

Filtered GRN: 19687 edges (from 91331), 88 TFs, 2000 target genes


,source,target,weight
0,Nr1h3,4930578g10rik,25.948450
1,Pax5,Gm28694,25.637556
2,Pax5,Ighv1-36,25.637556
3,Sox4,Nr5a2,25.501062
4,Bcl11b,4933406j10rik,25.482482
5,Id3,Ppp1r13l,24.456386
6,Id3,Ntn4,24.456386
7,Id3,Trbv1,24.456386
8,Junb,Trnp1,21.691888
9,Mef2c,1700028i16rik,21.596771


---

## Save the GRN

Save the inferred GRN for use in other ReCoN tutorials (e.g., Tutorial 2: Multicellular Coordination).

In [8]:
# Save to CSV
filtered_grn.to_csv("recon_grnboost_grn.csv", index=False)
print("GRN saved to recon_grnboost_grn.csv")

GRN saved to recon_grnboost_grn.csv


```{tip}
**Next steps**

Use this GRN in:
- [Tutorial 2: Multicellular Coordination](2.recon_multicellular_coordination.ipynb) - Explore upstream regulators
- [Tutorial 3: Molecular Cascades](3.recon_molecular_cascades.ipynb) - Visualize signaling pathways

Have scATAC-seq data available? Rebuild this GRN with chromatin accessibility evidence in [Tutorial 4: Building GRNs with HuMMuS](4.recon_hummus.ipynb) for higher-precision TF-gene links.
```

---

This work uses **ReCoN**'s wrapper of [**arboreto¹**](#arboreto), which reimplements the GRNBoost2 algorithm.

---

### References
<a id="arboreto"></a>
[1] Moerman, T., Aibar, S., Bravo González-Blas, C., Simm, J., Moreau, Y., Aerts, J., & Aerts, S. (2019). GRNBoost2 and Arboreto: efficient and scalable inference of gene regulatory networks. *Bioinformatics*, 35(12), 2159-2161.